# Data preparation, first transformations and EDA

In [1]:
pip install currencyconverter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 617.5/617.5 kB 5.2 MB/s eta 0:00:00


In [2]:
import pandas as pd
import chardet
from matplotlib import pyplot as plt
import seaborn as sns
import warnings
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from currency_converter import CurrencyConverter
from datetime import date
from scipy import stats
warnings.filterwarnings("ignore")
pd.options.display.float_format = '{:.2f}'.format

In [3]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Wise.csv', encoding = 'Latin-1')

I tried several encodings here, as standard UTF-8 was not working. Others were returning wrong number of rows, and i know there shoudl be 100k of them.

In [4]:
df.head()

,user_id,request_id,target_recipient_id,date_user_created,addr_country_code,addr_city,recipient_country_code,flag_personal_business,payment_type,date_request_submitted,...,ccy_target,transfer_to_self,sending_bank_name,sending_bank_country,payment_reference_classification,device,transfer_sequence,days_since_previous_req,first_attempt_date,first_success_date
0,77656618388e134648db01eecf7e79ee,2e9f911150e3a79e8d71a35779706e4c,992e0a729d6380d3b50aef5aa7c22572,27/01/2014 15:02,DEU,Berlin,GB,Personal,Direct Debit,26/08/2016 07:35,...,GBP,Other Recipient,Other/unknown,Other/unknown,gift,Desktop Web,55.00,0.00,27/01/2014 16:01,27/01/2014 16:17
1,a2497e0c763a7e5640fbf05e53fe0466,69cdf2f9ab2f59d10b636dc86bc9d7b7,02878ea857dbc90b2ed89b8f3488d501,12/10/2015 15:35,CAN,toronto,US,Personal,NaN,23/10/2016 22:54,...,USD,Other Recipient,Other/unknown,Other/unknown,expense,iOS App,1.00,NaN,23/10/2016 22:54,27/10/2016 16:45
2,759735d092819085c125a5cf81faf24b,5d7d30d709268f22f1a73ddbd6601690,927d3808cdc31d61226ae7c80bc8de16,04/10/2016 11:42,GBR,bolton,PT,Personal,Bank Transfer,26/10/2016 13:42,...,EUR,Self-recipient: Exact name match,NATIONAL WESTMINSTER BANK PLC,GB,blank,Android App,10.00,1.00,04/10/2016 12:27,04/10/2016 16:59
3,df9627db375322e65f4648ca72f4c630,0df2fc0a4a31595678cd1de3fad57e15,0411b7eb4c220a14876e77da8125f79b,17/10/2014 00:27,GBR,swindon,IN,Personal,Cards,28/01/2015 23:36,...,INR,Self-recipient: Email match,Other/unknown,Other/unknown,loan,Mobile Web,9.00,2.00,19/10/2014 22:00,24/11/2014 07:32
4,5672b2f16063ed75fbb304fee57c024b,7dcbf32659ae5ef61e10e5174a314d7d,dc248a1266709a71e45c40f33056bbb1,12/08/2015 07:45,FRA,paris,GB,Personal,Cards,18/08/2015 08:55,...,GBP,Other Recipient,LA BANQUE POSTALE,FR,blank,Desktop Web,1.00,NaN,18/08/2015 08:55,18/08/2015 09:44


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 28 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   user_id                           99998 non-null   object 
 1   request_id                        100000 non-null  object 
 2   target_recipient_id               100000 non-null  object 
 3   date_user_created                 99998 non-null   object 
 4   addr_country_code                 100000 non-null  object 
 5   addr_city                         99994 non-null   object 
 6   recipient_country_code            99991 non-null   object 
 7   flag_personal_business            99988 non-null   object 
 8   payment_type                      81217 non-null   object 
 9   date_request_submitted            99988 non-null   object 
 10  date_request_received             78610 non-null   object 
 11  date_request_transferred          77372 non-null   ob

In [6]:
df_fixed = df

In [7]:
for col in df_fixed.columns[[13, 14, 15]]:
    df_fixed[col] = pd.to_numeric(df_fixed[col], errors='coerce')

Here i am transforming obvious numeric columns from object type

In [8]:
for col in ['date_user_created', 'date_request_submitted', 'date_request_received', 'first_attempt_date',	'first_success_date']:
    df_fixed[col] = pd.to_datetime(df_fixed[col], format='%d/%m/%Y %H:%M', errors='coerce')

This function will replace NaN to NaT

In [9]:
df_fixed.head()

,user_id,request_id,target_recipient_id,date_user_created,addr_country_code,addr_city,recipient_country_code,flag_personal_business,payment_type,date_request_submitted,...,ccy_target,transfer_to_self,sending_bank_name,sending_bank_country,payment_reference_classification,device,transfer_sequence,days_since_previous_req,first_attempt_date,first_success_date
0,77656618388e134648db01eecf7e79ee,2e9f911150e3a79e8d71a35779706e4c,992e0a729d6380d3b50aef5aa7c22572,2014-01-27 15:02:00,DEU,Berlin,GB,Personal,Direct Debit,2016-08-26 07:35:00,...,GBP,Other Recipient,Other/unknown,Other/unknown,gift,Desktop Web,55.00,0.00,2014-01-27 16:01:00,2014-01-27 16:17:00
1,a2497e0c763a7e5640fbf05e53fe0466,69cdf2f9ab2f59d10b636dc86bc9d7b7,02878ea857dbc90b2ed89b8f3488d501,2015-10-12 15:35:00,CAN,toronto,US,Personal,NaN,2016-10-23 22:54:00,...,USD,Other Recipient,Other/unknown,Other/unknown,expense,iOS App,1.00,NaN,2016-10-23 22:54:00,2016-10-27 16:45:00
2,759735d092819085c125a5cf81faf24b,5d7d30d709268f22f1a73ddbd6601690,927d3808cdc31d61226ae7c80bc8de16,2016-10-04 11:42:00,GBR,bolton,PT,Personal,Bank Transfer,2016-10-26 13:42:00,...,EUR,Self-recipient: Exact name match,NATIONAL WESTMINSTER BANK PLC,GB,blank,Android App,10.00,1.00,2016-10-04 12:27:00,2016-10-04 16:59:00
3,df9627db375322e65f4648ca72f4c630,0df2fc0a4a31595678cd1de3fad57e15,0411b7eb4c220a14876e77da8125f79b,2014-10-17 00:27:00,GBR,swindon,IN,Personal,Cards,2015-01-28 23:36:00,...,INR,Self-recipient: Email match,Other/unknown,Other/unknown,loan,Mobile Web,9.00,2.00,2014-10-19 22:00:00,2014-11-24 07:32:00
4,5672b2f16063ed75fbb304fee57c024b,7dcbf32659ae5ef61e10e5174a314d7d,dc248a1266709a71e45c40f33056bbb1,2015-08-12 07:45:00,FRA,paris,GB,Personal,Cards,2015-08-18 08:55:00,...,GBP,Other Recipient,LA BANQUE POSTALE,FR,blank,Desktop Web,1.00,NaN,2015-08-18 08:55:00,2015-08-18 09:44:00


In [10]:
df_fixed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 28 columns):
 #   Column                            Non-Null Count   Dtype         
---  ------                            --------------   -----         
 0   user_id                           99998 non-null   object        
 1   request_id                        100000 non-null  object        
 2   target_recipient_id               100000 non-null  object        
 3   date_user_created                 99993 non-null   datetime64[ns]
 4   addr_country_code                 100000 non-null  object        
 5   addr_city                         99994 non-null   object        
 6   recipient_country_code            99991 non-null   object        
 7   flag_personal_business            99988 non-null   object        
 8   payment_type                      81217 non-null   object        
 9   date_request_submitted            99985 non-null   datetime64[ns]
 10  date_request_received            

Several columns have a substantial number of missing values. `payment_type`, `date_request_received`, `date_request_transferred`, `date_request_cancelled`, `invoice_value`, and `invoice_value_cancel` stand out.

 Columns like `user_id`, `addr_city`, `recipient_country_code`, etc., have a small number of missing values.

In [11]:
print(df_fixed.isnull().sum())

user_id                                 2
request_id                              0
target_recipient_id                     0
date_user_created                       7
addr_country_code                       0
addr_city                               6
recipient_country_code                  9
flag_personal_business                 12
payment_type                        18783
date_request_submitted                 15
date_request_received               21398
date_request_transferred            22628
date_request_cancelled              77736
invoice_value                       22275
invoice_value_cancel                77744
flag_transferred                       15
payment_status                          8
ccy_send                                8
ccy_target                              8
transfer_to_self                        8
sending_bank_name                      10
sending_bank_country                    8
payment_reference_classification        8
device                            

Missing `user_id` seems to be an important issue. `date_user_created` - the same. I guess, the best way would be to delete those rows without them. Anyway, loosing 7 rows seems not to be a critical thing considering we have 99993 left

Also, from the top of my mind -


```
 0   user_id                           99993 non-null  object        
 1   request_id                        99993 non-null  object        
 2   target_recipient_id               99993 non-null  object        
 3   date_user_created                 99993 non-null  datetime64[ns]
 4   addr_country_code                 99993 non-null  object        
 5   addr_city                         99989 non-null  object        
 6   recipient_country_code            99986 non-null  object        
 7   flag_personal_business            99986 non-null  object
```

Here we missing a few values for `addr_city`, `recipient_country_code`, `flag_personal_business`. This might be an indicator for transaction failiur. But that must be checked











In [12]:
df_fixed.dropna(subset=['user_id', 'date_user_created'], how='any', inplace=True)

In [13]:
columns_to_drop = [
    'addr_city',
    'recipient_country_code',
    'flag_personal_business',
    'flag_transferred',
    'transfer_sequence',
    'date_request_submitted',
    'payment_status',
    'ccy_send',
    'transfer_to_self',
    'sending_bank_name',
    'sending_bank_country',
    'payment_reference_classification',
    'device',
    'first_attempt_date'
]

for col in columns_to_drop:
    df_fixed.dropna(subset=[col], inplace=True)


print(df_fixed.isnull().sum())

user_id                                 0
request_id                              0
target_recipient_id                     0
date_user_created                       0
addr_country_code                       0
addr_city                               0
recipient_country_code                  0
flag_personal_business                  0
payment_type                        18773
date_request_submitted                  0
date_request_received               21382
date_request_transferred            22619
date_request_cancelled              77724
invoice_value                       22258
invoice_value_cancel                77724
flag_transferred                        0
payment_status                          0
ccy_send                                0
ccy_target                              0
transfer_to_self                        0
sending_bank_name                       0
sending_bank_country                    0
payment_reference_classification        0
device                            

Now it looks much better. The only thing that is left and can be modified to my mind is payment types

In [14]:
df_fixed['payment_type'].fillna('Unknown', inplace=True)

In [15]:
df_fixed.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99980 entries, 0 to 99999
Data columns (total 28 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   user_id                           99980 non-null  object        
 1   request_id                        99980 non-null  object        
 2   target_recipient_id               99980 non-null  object        
 3   date_user_created                 99980 non-null  datetime64[ns]
 4   addr_country_code                 99980 non-null  object        
 5   addr_city                         99980 non-null  object        
 6   recipient_country_code            99980 non-null  object        
 7   flag_personal_business            99980 non-null  object        
 8   payment_type                      99980 non-null  object        
 9   date_request_submitted            99980 non-null  datetime64[ns]
 10  date_request_received             78598 non-null  d

In [16]:
c = CurrencyConverter()
def convert_to_usd_currencyconverter(df):

    df['invoice_value_usd'] = np.nan

    for index, row in df.iterrows():
        transaction_date = row['date_request_submitted']
        currency = row['ccy_send']
        amount = row['invoice_value']

        if pd.isna(transaction_date) or pd.isna(currency) or pd.isna(amount):
            continue

        if currency == 'USD':
            df.loc[index, 'invoice_value_usd'] = amount
            continue

        try:
            conversion_date = transaction_date.date()
            usd_amount = c.convert(amount, currency, 'USD', date=conversion_date)
            df.loc[index, 'invoice_value_usd'] = usd_amount
        except Exception as e:
            print(f"Error converting {amount} {currency} on {transaction_date}: {e}")


    return df

In [17]:
convert_to_usd_currencyconverter(df_fixed)

Выходные данные были обрезаны до нескольких последних строк (5000).
Error converting 241.81 GBP on 2016-02-13 14:57:00: GBP has no rate for 2016-02-13
Error converting 150.0 GBP on 2015-10-18 11:37:00: GBP has no rate for 2015-10-18
Error converting 255.96 GBP on 2015-07-25 06:31:00: GBP has no rate for 2015-07-25
Error converting 518.0352 EUR on 2016-08-13 14:38:00: USD has no rate for 2016-08-13
Error converting 40.09 GBP on 2016-07-03 21:04:00: GBP has no rate for 2016-07-03
Error converting 421.772 EUR on 2014-04-20 22:16:00: USD has no rate for 2014-04-20
Error converting 3408.872 EUR on 2016-08-27 14:39:00: USD has no rate for 2016-08-27
Error converting 208.97 GBP on 2016-05-15 06:25:00: GBP has no rate for 2016-05-15
Error converting 603.01 GBP on 2016-11-13 07:54:00: GBP has no rate for 2016-11-13
Error converting 2345.81 GBP on 2015-08-09 18:55:00: GBP has no rate for 2015-08-09
Error converting 1728.3648 EUR on 2015-12-06 16:13:00: USD has no rate for 2015-12-06
Error conver

,user_id,request_id,target_recipient_id,date_user_created,addr_country_code,addr_city,recipient_country_code,flag_personal_business,payment_type,date_request_submitted,...,transfer_to_self,sending_bank_name,sending_bank_country,payment_reference_classification,device,transfer_sequence,days_since_previous_req,first_attempt_date,first_success_date,invoice_value_usd
0,77656618388e134648db01eecf7e79ee,2e9f911150e3a79e8d71a35779706e4c,992e0a729d6380d3b50aef5aa7c22572,2014-01-27 15:02:00,DEU,Berlin,GB,Personal,Direct Debit,2016-08-26 07:35:00,...,Other Recipient,Other/unknown,Other/unknown,gift,Desktop Web,55.00,0.00,2014-01-27 16:01:00,2014-01-27 16:17:00,175.91
1,a2497e0c763a7e5640fbf05e53fe0466,69cdf2f9ab2f59d10b636dc86bc9d7b7,02878ea857dbc90b2ed89b8f3488d501,2015-10-12 15:35:00,CAN,toronto,US,Personal,Unknown,2016-10-23 22:54:00,...,Other Recipient,Other/unknown,Other/unknown,expense,iOS App,1.00,NaN,2016-10-23 22:54:00,2016-10-27 16:45:00,NaN
2,759735d092819085c125a5cf81faf24b,5d7d30d709268f22f1a73ddbd6601690,927d3808cdc31d61226ae7c80bc8de16,2016-10-04 11:42:00,GBR,bolton,PT,Personal,Bank Transfer,2016-10-26 13:42:00,...,Self-recipient: Exact name match,NATIONAL WESTMINSTER BANK PLC,GB,blank,Android App,10.00,1.00,2016-10-04 12:27:00,2016-10-04 16:59:00,6105.74
3,df9627db375322e65f4648ca72f4c630,0df2fc0a4a31595678cd1de3fad57e15,0411b7eb4c220a14876e77da8125f79b,2014-10-17 00:27:00,GBR,swindon,IN,Personal,Cards,2015-01-28 23:36:00,...,Self-recipient: Email match,Other/unknown,Other/unknown,loan,Mobile Web,9.00,2.00,2014-10-19 22:00:00,2014-11-24 07:32:00,227.91
4,5672b2f16063ed75fbb304fee57c024b,7dcbf32659ae5ef61e10e5174a314d7d,dc248a1266709a71e45c40f33056bbb1,2015-08-12 07:45:00,FRA,paris,GB,Personal,Cards,2015-08-18 08:55:00,...,Other Recipient,LA BANQUE POSTALE,FR,blank,Desktop Web,1.00,NaN,2015-08-18 08:55:00,2015-08-18 09:44:00,663.74
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,c39e585ef0bb093e423a75e8a3d74c3d,2d9f3930cf83ab3fb1d6a14b42b2c6ce,c21b7d42d0cb67cbf5ce0a479f6dbac4,2014-03-10 12:11:00,GBR,Wembley,HU,Personal,Cards,2014-10-20 21:14:00,...,Other Recipient,Other/unknown,Other/unknown,blank,Desktop Web,3.00,3.00,2014-10-02 13:16:00,2014-10-02 14:12:00,30.67
99996,52c3a9f1e015635324743087c740c8a8,5a7eda59e8695c87253d7cece9665fcc,3056e69236ff441c0b8b8c0e09f255a2,2015-11-19 20:03:00,GBR,Peterhead,PL,Personal,Cards,2016-04-15 05:22:00,...,Self-recipient: Exact name match,ROYAL BANK OF SCOTLAND PLC,GB,blank,Mobile Web,21.00,6.00,2015-11-19 20:25:00,2015-12-14 06:44:00,283.61
99997,068fd3da986103729a9c558d217ca66c,73b43e68e21a3cda595b4204175adb1c,8aa083e482ef8fffc1dc57aae9df4098,2015-07-08 05:48:00,USA,Boulder,DE,Personal,Unknown,2016-06-30 11:54:00,...,Self-recipient: Exact name match,Other/unknown,Other/unknown,blank,Desktop Web,11.00,0.00,2015-07-08 05:55:00,2016-01-20 07:22:00,NaN
99998,cd80a491a39f84f0cb9e9fda6c7de95a,f59cab15bb69973e95a9583f17375023,56ded1634c35d7f9006938c0fa0ffa73,2015-05-01 08:17:00,AUS,Sydney,GB,Personal,Unknown,2016-04-28 04:22:00,...,"Family (Last Matches, 1st name different)",Other/unknown,Other/unknown,blank,Mobile Web,17.00,11.00,2015-05-01 08:52:00,2015-05-05 10:09:00,NaN


In [18]:
df_fixed.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99980 entries, 0 to 99999
Data columns (total 29 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   user_id                           99980 non-null  object        
 1   request_id                        99980 non-null  object        
 2   target_recipient_id               99980 non-null  object        
 3   date_user_created                 99980 non-null  datetime64[ns]
 4   addr_country_code                 99980 non-null  object        
 5   addr_city                         99980 non-null  object        
 6   recipient_country_code            99980 non-null  object        
 7   flag_personal_business            99980 non-null  object        
 8   payment_type                      99980 non-null  object        
 9   date_request_submitted            99980 non-null  datetime64[ns]
 10  date_request_received             78598 non-null  d

In [19]:
df_fixed['request_id'].nunique()

99980

Seems like all transactions are unique in the dataset

Lets also trim down the dataset, as we have a lot of variations in it and just keeping only most frequent combination will save us a lot of efforts.

In [20]:
countries = df_fixed.groupby(['addr_country_code']).agg({'request_id': 'count'}).reset_index().sort_values(by=['request_id'], ascending=False)
countries

,addr_country_code,request_id
46,GBR,52505
142,USA,8305
40,ESP,5631
33,DEU,5158
44,FRA,3609
...,...,...
102,NPL,1
112,PSE,1
121,SLE,1
122,SLV,1


only 15 countries have more than 1000 transactions.

In [21]:
df_trimmed = df_fixed[df_fixed['addr_country_code'].isin(countries.addr_country_code[countries['request_id'] > 700])]

In [22]:
df_loss = df_fixed[~df_fixed['addr_country_code'].isin(countries.addr_country_code[countries['request_id'] > 700])]
df_loss.invoice_value_usd.sum()

5224809.954277166

In [23]:
df_loss.invoice_value_usd.sum() / df_trimmed.invoice_value_usd.sum()

0.04790921210494551

I decided to stop on threshhold of 700 transactions per country, so we lose only 5% of all data.

In [24]:
df_trimmed.groupby(['payment_reference_classification']).agg({'request_id': 'count'}).reset_index().sort_values(by=['request_id'], ascending=False).head(50)

,payment_reference_classification,request_id
2,blank,46217
0,Other/unknown,26281
12,invoice,4974
14,monthly,3491
7,family,1975
17,rent,1464
9,generic,1325
20,self_transfer,963
10,gift,801
11,house,763


Here nothing very strange, the only thing worth noticing - we have a lot of transactions that are not properly classified. Maybe could be important to add this as mandatory field during the process.

In [25]:
payments = df_trimmed.groupby(['payment_type']).agg({'request_id': 'count'}).reset_index().sort_values(by=['request_id'], ascending=False).head(50)
payments

,payment_type,request_id
3,Cards,44191
1,Bank Transfer,24737
12,Unknown,17043
4,Direct Debit,5650
2,Boleto,977
9,Poli,287
5,Insta Debit,275
11,Trustly,76
8,Number26,65
0,Apple Pay - Adyen,20


Okay, here we can eliminate any type of payment, which has less than 100 transactions, it will be <1%

In [26]:
df_trimmed = df_trimmed[df_trimmed['payment_type'].isin(payments.payment_type[payments['request_id'] > 100])]

In [27]:
df_trimmed.groupby(['device']).agg({'request_id': 'count'}).reset_index().sort_values(by=['request_id'], ascending=False).head(50)

,device,request_id
1,Desktop Web,64974
3,iOS App,13599
0,Android App,7672
2,Mobile Web,6915


Most of transactions are done in the browsers of PCs, also nothing really pointing on outliers

So, considering that not all invoices were properly conveted, estimated amount loss of data is %5, which is about 5 mil USD. With average loss rates about 1% that makes around 50000 USD of allowed losses.

In [28]:
df_trimmed['date_request_submitted'] = pd.to_datetime(df_trimmed['date_request_submitted'])
df_trimmed['date_request_received'] = pd.to_datetime(df_trimmed['date_request_received'])
df_trimmed['date_request_transferred'] = pd.to_datetime(df_trimmed['date_request_transferred'], errors='coerce')

df_trimmed['time_to_receive'] = df_trimmed['date_request_received'] - df_trimmed['date_request_submitted']
df_trimmed['time_to_transfer'] = df_trimmed['date_request_transferred'] - df_trimmed['date_request_received']

In [29]:
df_trimmed.loc[df_trimmed['date_request_submitted'] > df_trimmed['date_request_received'], 'date_request_submitted'].count()

178

Those 178 rows i suggest dropping


In [30]:
df_trimmed = df_trimmed[~(df_trimmed['date_request_submitted'] > df_trimmed['date_request_received'])]

In [31]:
df_trimmed.loc[df_trimmed['date_request_submitted'] > df_trimmed['date_request_transferred'], 'date_request_submitted'].count()

16013

This is very odd behaviour and i suppose nothing should work like this. As i dont know something, i would suggest to keep is as it is, because 16% of all dataset is huge...

## Customer Classification Analysis

This analysis categorizes customers into three distinct groups based on their transaction history and recipient behavior, aiming to identify potential risk patterns:

**1. New - Never Received Money:**

* **Description:** Customers making their first transaction and whose recipients have not previously received funds through the platform.
* **Potential Implications:**
    * May represent genuine new users.
    * Could also indicate potential fraud attempts, where new accounts are created to test the system or move illicit funds to previously unknown recipients.
    * Might be used for "smurfing" purposes, where small amounts are transferred to avoid detection.
* **Monitoring:** Track transaction patterns closely, especially for unusual volumes or values.

**2. New - Received Money:**

* **Description:** Customers making their first transaction, but their recipients have previously received funds through the platform.
* **Potential Implications:**
    * Could be legitimate transactions to established recipients.
    * May also suggest attempts to obscure transaction origins by using previously known recipients.
    * Potentially indicate a network of related accounts used for illicit activities.
* **Monitoring:** Investigate the relationship between the new user and the recipient, and monitor for unusual transaction patterns.

**3. Returning:**

* **Description:** Customers making their second or subsequent transactions.
* **Potential Implications:**
    * Represents established customers.
    * However, even returning customers can engage in fraudulent activities.
    * Changes in their transaction patterns should be monitored for anomalies.
* **Monitoring:** Analyze for deviations from established transaction behaviors, and investigate any sudden changes in volume, value, or recipient patterns.

**Overall Goal:**

This classification helps identify potential risk factors associated with different customer behaviors. By monitoring these groups, we can detect anomalies, mitigate risks, and improve the security of the transaction platform.

In [32]:
df_trimmed = df_trimmed.sort_values(by=['user_id', 'date_request_submitted'])
df_trimmed['customer_type'] = df_trimmed.groupby('user_id').cumcount() + 1

received_users = {}
for index,row in df_trimmed.iterrows():
  if row['target_recipient_id'] not in received_users:
    received_users[row['target_recipient_id']] = row['date_request_submitted']

def check_received(row):
    user_id = row['user_id']
    recipient_id = row['target_recipient_id']
    request_date = row['date_request_submitted']
    if recipient_id in received_users:
        if received_users[recipient_id] < request_date:
            return True
    return False

df_trimmed['received_money_once'] = df_trimmed.apply(check_received, axis=1)

def categorize_customer(row):
    if row['customer_type'] == 1:
        if row['received_money_once']:
            return 'New - Received Money'
        else:
            return 'New - Never Received Money'
    else:
        return 'Returning'

df_trimmed['customer_category'] = df_trimmed.apply(categorize_customer, axis=1)


In [33]:
df_trimmed.groupby(['customer_category']).agg({'request_id': 'count'}).reset_index().sort_values(by=['request_id'], ascending=False)

,customer_category,request_id
0,New - Never Received Money,83088
2,Returning,9873
1,New - Received Money,21


### Data Field Descriptions

This section provides detailed descriptions of each data field available in the dataset. Understanding these fields is essential for proper analysis and interpretation of the data.

#### Customer and Request Information

*   **`user_id`**: A unique identifier assigned to each customer. This field allows for the tracking of individual customer behavior across multiple requests.
*   **`request_id`**: A unique identifier for each transfer request initiated by a customer. This enables the tracking of individual transfers through different processing stages.
*   **`target_recipient_id`**: A unique identifier for the recipient of the transfer. This field is used to link transfers to specific recipients and track their history.
*   **`date_user_created`**: The date and time at which the customer's account was created. This information can be used to analyze customer tenure and lifecycle.

#### Sender and Recipient Details

*   **`addr_country_code`**: The country code of the sender's address, using a standardized two-letter format (e.g., "US" for the United States).
*   **`addr_city`**: The city where the sender is located.
*   **`recipient_country_code`**: The country code of the recipient's location, using a standardized two-letter format.
*   **`flag_personal_business`**: A flag indicating whether the transfer is for personal or business purposes. This could be a boolean value (True/False) or a categorical value (Personal/Business).
*   **`transfer_to_self`**: Indicates the type of recipient. Possible values can be: self, other.

#### Payment and Transfer Details

*   **`payment_type`**: The method used by the customer to fund the transfer (e.g., bank transfer, credit card, debit card).
*   **`date_request_submitted`**: The date and time at which the customer initiated or set up the transfer request.
*   **`date_request_received`**: The date and time at which the customer's funds were received and confirmed by our system.
*   **`date_request_transferred`**: The date and time at which the funds were successfully paid out to the recipient.
*   **`invoice_value`**: The total amount of money the customer is sending, before any fees or exchange rate adjustments.
*   **`flag_transferred`**: A flag indicating whether the transfer was successfully completed. This is typically a boolean value (True/False).
*   **`payment_status`**: The current status of the payment. Examples include "Pending", "Completed", "Failed", "Refunded".
*   **`ccy_send`**: The currency from which the customer is sending money (e.g., "USD" for US Dollars, "EUR" for Euros).
*   **`ccy_target`**: The currency in which the customer's recipient will receive the money.
*   **`sending_bank_name`**: The name of the bank from which the customer is sending the funds.
*   **`sending_bank_country`**: The country in which the sending bank is located.
*   **`payment_reference_classification`**: This indicates the reason for the transfer, as entered by the customer. This may include categories like "Family Support", "Invoice Payment", "Personal Expenses", etc.

#### Customer Activity and Device

*   **`device`**: The platform or device used by the customer to initiate the transfer (e.g., "Web", "iOS", "Android").
*   **`transfer_sequence`**: A sequential number indicating how many transfers the customer has made using our system. The first transfer will have a value of 1, the second will have a value of 2, and so on.
*   **`days_since_previous_req`**: The number of days between the current transfer request and the customer's immediately previous transfer request.
*   **`First_attempt_date`**: The date and time of the first attempt to process the transfer.
*   **`first_success_date`**: The date and time of the first successful transfer to the recipient.


# Data investigation

## Potential System Vulnerability Indicators

**Focus:** Identifying anomalies and patterns in transaction data that may signal risks.

**1. Country-Based Anomalies:**

* **Hypothesis:** Unusual transaction patterns exist based on sender/recipient country.
* **Analysis:**
    * Analyze transaction volume and value distributions by `addr_country_code` and `recipient_country_code`.
    * Identify countries with unusually high/low transfer volumes or average invoice values.
    * Examine currency exchange patterns (`ccy_send`, `ccy_target`) for specific country pairs.
* **Why this might be an issue:**
    * **High volumes from specific countries:** Could indicate money laundering hotspots or regions with high fraud rates.
    * **Unusual exchange patterns:** May signal attempts to circumvent currency controls or obscure transaction origins.
    * **Unexpectedly high average invoice values:** Could show that a specific country is being used for high value fraudulent transactions.

**2. Payment Type Variations:**

* **Hypothesis:** Certain `payment_type` methods exhibit atypical transaction patterns.
* **Analysis:**
    * Analyze transfer success rates (`flag_transferred`, `payment_status`) by `payment_type`.
    * Compare average `invoice_value` and transfer processing times (`date_request_received`, `date_request_transferred`) across payment types.
    * Identify payment types with high rates of failed or pending transactions.
* **Why this might be an issue:**
    * **High failure rates for specific payment types:** May indicate vulnerabilities in those payment methods or their susceptibility to fraud.
    * **Significantly longer processing times:** Could suggest delays caused by security checks or attempts to delay detection.
    * **Payment types with high average invoice values:** Could indicate that those payment types are being used to move large amounts of fraudulent money.

**3. Amount-Based Irregularities:**

* **Hypothesis:** Transaction amounts reveal suspicious patterns.
* **Analysis:**
    * Analyze the distribution of `invoice_value` and identify outliers.
    * Examine the relationship between `invoice_value` and `payment_type` or `device`.
    * Look for patterns of many small transactions from the same user.
* **Why this might be an issue:**
    * **Outliers:** Could be evidence of fraudulent transactions or data entry errors.
    * **Patterns of small transactions:** Might be "smurfing" (structuring), a tactic used to avoid reporting thresholds for large transactions.
    * **Unusual invoice value to payment type or device relationships:** Could demonstrate that a specific device or payment type is being used for high value fraud.

**4. Device and User Behavior:**

* **Hypothesis:** Device and user activity patterns indicate potential risks.
* **Analysis:**
    * Analyze transaction frequency (`transfer_sequence`, `days_since_previous_req`) by `device`.
    * Examine the distribution of `transfer_to_self` and `flag_personal_business` by device.
    * Analyze the time difference between `date_user_created` and the first transaction, and then the difference between the first attempt date and the first success date.
* **Why this might be an issue:**
    * **High transaction frequency from a single device:** Could indicate automated or bot-driven fraud.
    * **High rates of transfers to self:** May be used to move money between accounts controlled by the same individual, potentially for money laundering.
    * **Short time from account creation to first transaction:** Could signal accounts created solely for fraudulent purposes.
    * **Large time between first attempt and success:** Could indicate that multiple attempts were made to bypass security.

**5. Transfer Speed and Status:**

* **Hypothesis:** Anomalies in transfer processing times and statuses may indicate issues.
* **Analysis:**
    * Analyze the time differences between `date_request_submitted`, `date_request_received`, and `date_request_transferred`.
    * Identify transactions with unusually long processing times or frequent status changes (`payment_status`).
    * Look at the time difference between the first attempt date and the first success date.
* **Why this might be an issue:**
    * **Unusually long processing times:** May suggest manual intervention required due to flagged transactions.
    * **Frequent status changes:** Could indicate attempts to manipulate the system or bypass security measures.
    * **Large time between first attempt and success:** Could indicate that multiple attempts were made to bypass security.

**6. Bank and Reference Analysis:**

* **Hypothesis:** Patterns in `sending_bank_name`, `sending_bank_country`, and `payment_reference_classification` may reveal risks.
* **Analysis:**
    * Analyze the distribution of transactions by `sending_bank_name` and `sending_bank_country`.
    * Identify banks with unusually high transaction volumes or atypical transfer patterns.
    * Analyze the frequency of different `payment_reference_classification` values and identify unusual patterns.
    * Analyze the frequency of different banks being used by the same user.
* **Why this might be an issue:**
    * **High transaction volumes from obscure banks:** Could indicate the use of shell banks or banks in high-risk jurisdictions.
    * **Unusual payment reference classifications:** May be used to obscure the true purpose of transactions.
    * **Users rapidly changing sending banks:** could indicate that the user is attempting to avoid detection, or that they are using stolen bank accounts.

**Key Considerations:**

* Focus on identifying anomalies and deviations from expected patterns.
* Utilize time-based analysis to detect trends and changes in behavior.
* Combine multiple variables to create more robust risk indicators.
* Data cleaning is very important.

### 1. Anomalies Check:

Here i want to check whether there are any strange data points in some of following columns

In [34]:
columns = ['addr_country_code', 'recipient_country_code', 'ccy_send', 'ccy_target', 'transfer_to_self', 'payment_reference_classification', 'device', 'payment_type']

for col in columns:
    print(f"Unique values in column '{col}':")
    print(df[col].unique())
    print()
    print('---------------')


Unique values in column 'addr_country_code':
['DEU' 'CAN' 'GBR' 'FRA' 'USA' 'POL' 'AUS' 'SWE' 'BRA' 'IRL' 'ESP' 'BEL'
 'ROU' 'NLD' 'SVK' 'IND' 'AUT' 'ITA' 'FIN' 'EST' 'PRT' 'CZE' 'NZL' 'BGR'
 'MYS' 'LVA' 'DNK' 'NOR' 'CHE' 'HUN' 'PHL' 'ASM' 'SGP' 'IDN' 'MLT' 'RUS'
 'GRL' 'SVN' 'NGA' 'GRC' 'TUR' 'IMN' 'CYP' 'ZAF' 'THA' 'VEN' 'LTU' 'CYM'
 'GIB' 'AIA' 'JPN' 'TUN' 'LUX' 'ARE' 'MAR' 'JEY' 'EGY' 'PRI' 'UKR' 'GGY'
 'HKG' 'ARG' 'MCO' 'BIH' 'COL' 'MEX' 'ISL' 'GHA' 'AND' 'SAU' 'KGZ' 'PAK'
 'CHL' 'KEN' 'PAN' 'IOT' 'TCA' 'CUW' 'GEO' 'BGD' 'MDA' 'KOR' 'SEN' 'HRV'
 'SPM' 'VGB' 'QAT' 'PER' 'DOM' 'ISR' 'SYC' 'VNM' 'MTQ' 'BLZ' 'SLV' 'ECU'
 'MUS' 'DZA' 'KWT' 'CRI' 'REU' 'CHN' 'SRB' 'OMN' 'GLP' 'ERI' 'TGO' 'LKA'
 'MDG' 'GAB' 'AZE' 'KAZ' 'TWN' 'NIC' 'LBN' 'NCL' 'BLM' 'BHR' 'MKD' 'JAM'
 'ETH' 'BOL' 'ALB' 'URY' 'PYF' 'MOZ' 'PSE' 'LAO' 'MAC' 'GTM' 'AGO' 'BLR'
 'MNE' 'SWZ' 'TZA' 'MDV' 'BHS' 'UGA' 'SXM' 'SMR' 'ZMB' 'BRN' 'KHM' 'SLE'
 'LBY' 'NPL' 'BMU' 'NFK']

---------------
Unique values in column 'recipient_c

Here i just check for unqiue values in those columns, where number of elements is not huge. In column `recipient_country_code` we can observe some strange destinations `EmptyRecipient`, `BangladeshLocalRecipient`, `EmailRecipient`, `SriLankaLocalRecipient`, `JapaneseLocal`, `VietnamEarthportRecipient`, `BalanceRecipient`. Considering that it should be 2 code name, it looks wrong (**`recipient_country_code`**: The country code of the recipient's location, using a standardized two-letter format.)

In [35]:
wrong_recipient_codes = ['EmptyRecipient', 'BangladeshLocalRecipient', 'EmailRecipient', 'SriLankaLocalRecipient', 'JapaneseLocal', 'VietnamEarthportRecipient', 'BalanceRecipient']

In [36]:
df_trimmed[df_trimmed['recipient_country_code'].isin(wrong_recipient_codes)].nunique()

,0
user_id,933
request_id,949
target_recipient_id,942
date_user_created,931
addr_country_code,19
addr_city,624
recipient_country_code,7
flag_personal_business,2
payment_type,7
date_request_submitted,949


We have 950 records using wrong recipient codes, where we have 934 unique `user_ids` and 943 `target_ids`

In [37]:
check = df_trimmed[df_trimmed['recipient_country_code'].isin(wrong_recipient_codes)]
print('total sum = ', check.invoice_value_usd.sum())
print('mean = ', check.invoice_value_usd.mean())
print('median = ', check.invoice_value_usd.median())

total sum =  117583.05037272317
mean =  1336.1710269627633
median =  323.01827939621546


This might be an issue, need to check status of these payments.

In [38]:
check.groupby(['payment_status']).agg({'request_id': 'count', 'invoice_value_usd': ('sum', 'mean', 'median')}).reset_index().sort_values(by=[('request_id', 'count')], ascending=False)

payment_status request_id invoice_value_usd               
                      count               sum    mean median
0      Cancelled        854              0.00     NaN    NaN
2    Transferred         88         112941.92 1394.34 323.34
1        Pending          7           4641.13  663.02 319.90

As expected, a lot of those were cancelled. I assume it was due to the issue in recipient country code. Lets also check, what is the amount that was transferred using these incorrect codes

In [39]:
check_1 = check[check['payment_status'] == 'Transferred']
check_2 = check[check['payment_status'] == 'Cancelled']
print('total sum = ', check_1.invoice_value_usd.sum())
print('mean = ', check_1.invoice_value_usd.mean())
print('median = ', check_1.invoice_value_usd.median())

total sum =  112941.92239783998
mean =  1394.3447209609874
median =  323.3354802887285


very interesting, that amount are pretty high there

In [40]:
check_2.groupby(['payment_type']).agg({'request_id': 'count', }).reset_index().sort_values(by=['request_id'], ascending=False)

,payment_type,request_id
4,Unknown,654
2,Cards,150
0,Bank Transfer,36
1,Boleto,10
3,Insta Debit,4


A lot of cancelled transcation have no payment type.

In [41]:
check_2.groupby(['payment_reference_classification']).agg({'request_id': 'count', }).reset_index().sort_values(by=['request_id'], ascending=False)

,payment_reference_classification,request_id
0,Other/unknown,359
2,blank,332
12,invoice,29
15,rent,21
19,test,19
7,family,16
14,monthly,15
10,gift,13
20,travel,9
1,bills,6


Also very interesting, that those transactions missing payment refference in most of cases

In [42]:
check_2.groupby(['addr_country_code']).agg({'request_id': 'count', }).reset_index().sort_values(by=['request_id'], ascending=False)

,addr_country_code,request_id
10,GBR,362
9,FRA,78
5,DEU,70
7,ESP,56
18,USA,54
14,ITA,30
3,CAN,28
0,AUS,24
8,EST,21
11,HUN,21


In terms of country split everything is prettyy standard - we have a lot of cancelled transactions in GBR, nut it is expected.

Just to confirm this statistics I want now to check, whether this trend is the same not only for cases with wrong recipient description, but also for all cases.

In [43]:
check_3 = df_trimmed[df_trimmed['payment_status'] == 'Cancelled']

In [44]:
graph = check_3.groupby(['payment_reference_classification']).agg({'request_id': 'count', }).reset_index().sort_values(by=['request_id'], ascending=False)
top_3_data = graph.head(3)
fig = px.bar(
    top_3_data,
    x='payment_reference_classification',
    y='request_id',
    title='Top 3 Cancelled Payment Reference Classifications by Request Count',
    labels={'request_id': 'Request Count', 'payment_reference_classification': 'Payment Reference Classification'}
)

fig.show()

Yes, indeed, trend is the same. I assume, that wrong payment refference decrease chances of transaction being approved.

## Visuals

In [45]:
def create_mean_plot_hue(df, time_granularity, filter_column, filter_values):

    filtered_df = df[df[filter_column].isin(filter_values)]


    time_series = filtered_df.set_index('date_request_submitted').groupby(
        [pd.Grouper(freq=time_granularity), filter_column]
    )


    time_series_agg = time_series['invoice_value_usd'].mean().reset_index(name='mean_value')


    fig = px.line(
        time_series_agg,
        x='date_request_submitted',
        y='mean_value',
        color=filter_column,
        title=f'Mean of Invoice Value USD ({time_granularity}) - {filter_column}',
        labels={'mean_value': 'Mean Value', 'date_request_submitted': 'Date', filter_column: filter_column}
    )


    if time_granularity == 'M':
        fig.update_xaxes(
            tickformat="%b %Y",
            dtick="M1"
        )

    fig.show()

In [46]:
def create_cancellation_rate_plot_hue(df, time_granularity, filter_column, filter_values):

    filtered_df = df[df[filter_column].isin(filter_values)]

    time_series = filtered_df.set_index('date_request_submitted').groupby(
        [pd.Grouper(freq=time_granularity), filter_column]
    )

    time_series_agg = time_series.agg(
        total_payments=('payment_status', 'count'),
        cancelled_payments=('payment_status', lambda x: (x == 'Cancelled').sum())
    )

    time_series_agg['cancellation_rate'] = time_series_agg['cancelled_payments'] / time_series_agg['total_payments']
    time_series_agg = time_series_agg.reset_index()

    fig = px.line(
        time_series_agg,
        x='date_request_submitted',
        y='cancellation_rate',
        color=filter_column,
        title=f'Cancellation Rate ({time_granularity}) - {filter_column}',
        labels={'cancellation_rate': 'Cancellation Rate', 'date_request_submitted': 'Date', filter_column: filter_column}
    )

    if time_granularity == 'M':
        fig.update_xaxes(
            tickformat="%b %Y",
            dtick="M1"
        )

    fig.show()

In [47]:
def create_outlier_rate_plot_5percent(df, time_granularity, filter_column, filter_values):

    filtered_df = df[df[filter_column].isin(filter_values)]


    time_series = filtered_df.set_index('date_request_submitted').groupby(
        [pd.Grouper(freq=time_granularity), filter_column]
    )

    def count_outliers(series):
        z_scores = stats.zscore(series, nan_policy='omit')
        return ((z_scores > 1.96) | (z_scores < -1.96)).sum()

    time_series_agg = time_series.agg(
        outlier_count=('invoice_value_usd', count_outliers),
        total_count=('invoice_value_usd', 'count')
    ).reset_index()

    time_series_agg['outlier_rate'] = time_series_agg['outlier_count'] / time_series_agg['total_count']

    fig = px.line(
        time_series_agg,
        x='date_request_submitted',
        y='outlier_rate',
        color=filter_column,
        title=f'Outlier Rate (5% Confidence) of Invoice Value USD ({time_granularity}) - {filter_column}',
        labels={'outlier_rate': 'Outlier Rate', 'date_request_submitted': 'Date', filter_column: filter_column}
    )

    if time_granularity == 'M':
        fig.update_xaxes(
            tickformat="%b %Y",
            dtick="M1"
        )

    fig.show()

In [48]:
def create_time_to_receive_days_lineplot(df, time_granularity, filter_column, filter_values):


    filtered_df = df[df[filter_column].isin(filter_values)]

    time_series = filtered_df.groupby([pd.Grouper(key='date_request_submitted', freq=time_granularity), filter_column])['time_to_receive'].mean().reset_index()
    time_series['time_to_receive_days'] = time_series['time_to_receive'].dt.total_seconds() / (60 * 60 * 24)

    fig = px.line(
        time_series,
        x='date_request_submitted',
        y='time_to_receive_days',
        color=filter_column,
        title=f'Mean Time to Receive (Days) Over Time ({time_granularity})',
        labels={'date_request_submitted': 'Date', 'time_to_receive_days': 'Mean Time to Receive (Days)', filter_column: filter_column}
    )

    if time_granularity == 'M':
        fig.update_xaxes(
            tickformat="%b %Y",
            dtick="M1"
        )

    fig.show()


### Countries

In [51]:
create_mean_plot_hue(df_trimmed, 'M', 'addr_country_code', ['GBR', 'USA', 'ESP', 'BEL', 'DEU'])
create_cancellation_rate_plot_hue(df_trimmed, 'M', 'addr_country_code', ['GBR', 'USA', 'FR', 'ESP', 'BEL', 'DEU'])
create_outlier_rate_plot_5percent(df_trimmed, 'M', 'addr_country_code', ['GBR', 'USA', 'ESP', 'BEL', 'DEU'])
create_time_to_receive_days_lineplot(df_trimmed, 'M', 'addr_country_code', ['GBR', 'USA', 'ESP', 'BEL', 'DEU'])

What we can say in general about the trends for mean (median and counts would be the same) of total transactions? mean flattens closer to 2016, this is due to increased amount number of transactions.

Over time cancellation rate becomes better, i guess this is caused by limitation at start of this bank program (more checks, less controll over data and so on). In the biggest countries we observe hure cancelation rates when all transcation start to appear, especcially in BEL and USA.

Ineteresting enough, we have a lot of outliers. Obviously, a lot of them are in GBR, but what is important - is the rate of those outliers to monthly number of transactions. In countries like BEL and DEU number of transactions is huge. Sometimes up to 20% and on average about 7-8%. While in GBR it is close to 1%, which is still significant, considering overall size of the market, but much better. Suprisingly USA hold a good spot with 2.5% only. And trend is pretty much the same: more established the market - the more stable situation with outlier rates.   

Worth noticing that time to receive is relatively long in DEU and in recent dates increased in BEL. As well as in USA significantly.

### payment type

In [52]:
column = 'payment_type'
filters = ['Direct Debit', 'Unknown' ,'Bank Transfer' ,'Cards' ,'Boleto', 'Insta Debit',
 'Poli' ,'Trustly', 'Manual Payment', 'Number26', 'Apple Pay - Adyen'
 'iDeal - Adyen', 'Kapcharge' ,'TW Balance Payment']
create_mean_plot_hue(df_trimmed, 'M', column, filters)
create_outlier_rate_plot_5percent(df_trimmed, 'M', column, filters)
create_cancellation_rate_plot_hue(df_trimmed, 'M', column, filters)
create_time_to_receive_days_lineplot(df_trimmed, 'M', column, filters)

Mostly canlelation rate is noticable only with one type - "Unknown", but that we might expect as well

Seems like the most established payment types are "Bank Transfer" and "Cards" and thus they have the lowest outlier rate. Also very long waiting time to receive funds for Boleto

### transfer to self

In [53]:
column = 'transfer_to_self'
filters = ['Other Recipient','Self-recipient: Exact name match',
 'Self-recipient: Email match', 'N.A. Sender or Recipient is business',
 'N.A. Recipient Email Unknown',
 'Family (Last Matches, 1st name different)', 'Self-recipient: Name match']
create_mean_plot_hue(df_trimmed, 'M', column, filters)
create_outlier_rate_plot_5percent(df_trimmed, 'M', column, filters)
create_cancellation_rate_plot_hue(df_trimmed, 'M', column, filters)
create_time_to_receive_days_lineplot(df_trimmed, 'M', column, filters)

Here i would outline one thing: 'Self-recipient: Name match' as identifier for transaction made to yourself shows a higher outlier rate in the recent dates. Might worth checking later what is going on there. Everythoing else looks stable

In [54]:
column = 'device'

filters = ['Desktop Web', 'iOS App' ,'Android App' ,'Mobile Web']
create_mean_plot_hue(df_trimmed, 'M', column, filters)
create_outlier_rate_plot_5percent(df_trimmed, 'M', column, filters)
create_cancellation_rate_plot_hue(df_trimmed, 'M', column, filters)
create_time_to_receive_days_lineplot(df_trimmed, 'M', column, filters)

What we can say about different devices: thends are the same. The strangest of them is "Android APP". Especially in the beginning.

### payment reference classification


In [55]:

column = 'payment_reference_classification'

filters = ['gift' ,'expense', 'blank', 'loan', 'invoice' ,'Other/unknown' ,'monthly'
 'house', 'family', 'salary' ,'travel', 'savings', 'wedding' ,'self_transfer',
 'education' ,'test', 'bills', 'rent', 'generic', 'deposit', 'taxes', 'freelance',
 'credit', 'mortgage' ,'pension']
create_mean_plot_hue(df_trimmed, 'M', column, filters)
create_outlier_rate_plot_5percent(df_trimmed, 'M', column, filters)
create_cancellation_rate_plot_hue(df_trimmed, 'M', column, filters)
create_time_to_receive_days_lineplot(df_trimmed, 'M', column, filters)

Okay, everything is not very clear due to huge amount of variables, but, what i see is that there are few categories, which appear frequntly as a top outliers for every metric i tracks: those are *taxes, freelance, test, education and mortgage*. Especially considering mean values. And the same can be reffred to waiting time to receive fund from sender. For Taxes it is very long  

## Customer types

In [56]:

column = 'customer_category'

filters = ['New - Never Received Money',
'Returning',
'New - Received Money']
create_mean_plot_hue(df_trimmed, 'M', column, filters)
create_outlier_rate_plot_5percent(df_trimmed, 'M', column, filters)
create_cancellation_rate_plot_hue(df_trimmed, 'M', column, filters)
create_time_to_receive_days_lineplot(df_trimmed, 'M', column, filters)

This suggests that returning customers are more likely to send higher amount in transaction. Also, pretty clear, that new users have higher rate for canclelation. Everything looks reasonable, but might lead the whole campaign to the state, where more strict limitations should be applied for new users.

**Summary of Transaction Trends and Potential Anomalies:**

**Overall Transaction Volume and Stabilization:**

* The mean (and likely median and count) of total transactions demonstrates a trend of flattening out closer to 2016. This stabilization is attributed to an increased volume of transactions, suggesting market maturity.

**Cancellation Rate Improvement:**

* The overall cancellation rate has improved over time, potentially due to the initial limitations of the bank's program. This includes better controls, more checks and better data handling.
* However, significant cancellation rate spikes are observed in major markets like BEL and USA during the program's early stages, which may indicate initial operational challenges.

**Outlier Analysis:**

* The *rate* of outliers relative to monthly transaction volume is concerning in BEL and DEU, reaching up to 20% in some instances, and averaging 7-8%.
* This is much higher than the 1% observed in GBR, and the 2.5% in USA.
* The trend indicates that more established markets exhibit greater stability in outlier rates.

**Processing Time (Time to Receive):**

* DEU consistently shows relatively long processing times for fund receipt.
* Recent increases in processing times are observed in BEL and, significantly, in the USA.

**Cancellation Rate by Payment Status:**

* The "Unknown" payment status category correlates with a noticeable increase in cancellation rates, as expected.

**Outlier Rate by Payment Type and Recipient:**

* "Bank Transfer" and "Cards" are the most established payment types, exhibiting the lowest outlier rates.
* "Boleto" payment type has very long waiting times.
* "Self-recipient: Name match" transactions show an increased outlier rate recently, warranting further investigation.

**Device-Specific Trends:**

* Transaction trends are generally consistent across devices, with "Android APP" showing some initial anomalies.

**Transaction Categories and Waiting Times:**

* Specific transaction categories (taxes, freelance, test, education, and mortgage) frequently appear as top outliers across various metrics, especially mean values.
* Transactions related to taxes exhibit particularly long waiting times for fund receipt.

**Customer Categorization and Trends:**

* Returning customers are more likely to send higher amounts in transactions.
* New users exhibit a higher cancellation rate.
* This suggests that more strict limitations should be applied for new users to mitigate potential risks.

**Key Takeaways and Potential Issues:**

* **Data Quality:** The high outlier rates and date inconsistencies suggest potential data quality issues that need to be addressed.
* **Operational Efficiency:** The varying processing times and cancellation rates across countries and payment types indicate potential operational inefficiencies or bottlenecks.
* **Risk Management:** The outlier patterns and unusual transaction categories highlight potential risks related to fraud, money laundering, or other illicit activities.
* **System Stability:** The initial cancellation rate spikes and device-specific anomalies may point to system stability issues during the program's early stages.
* **Monitoring:** The recent rise of outliers within self-recipient transactions should be monitored closely.
* **New User Risk:** The higher cancellation rates and potentially higher risk associated with new users necessitate the implementation of stricter limitations or enhanced verification procedures for this customer segment.

# Montly losses estimation

## Average Monthly Loss Estimation Logic (Logarithmic Scale)

This code estimates potential financial losses based on several identified risk factors within transaction data, presenting the results as a bar graph with a logarithmic y-axis. Here's a concise breakdown of the logic:

1.  **Data Filtering:**
    * The code filters the transaction data to include only records within a specified date range.

2.  **Risk Factor Analysis:**
    * **Unknown/Blank Payment Reference Classification:** Calculates the total `invoice_value_usd` for transactions with "Other/unknown" or "blank" payment reference classifications and multiplied by estimation of fee taken for every transaction (4%).
    * **Outliers:** Identifies and calculates 2.5% of the total `invoice_value_usd` for transactions considered outliers based on a Z-score threshold. This represents a potential loss due to risk from outliers and multiplied by estimation of fee taken for every transaction (4%).
    * **Self-Recipient Transactions:** Calculates 2.5% of losses associated with "Self-recipient: Name match" transactions that are also outliers and multiplied by estimation of fee taken for every transaction (4%).
    * **BEL/DEU Transactions:** Calculates 2.5% of losses from outlier transactions originating from Belgium (BEL) and Germany (DEU) and multiplied by estimation of fee taken for every transaction (4%).

3.  **Monthly Averaging:**
    * Calculates the number of months within the specified date range.
    * Divides each loss category's total value by the number of months to obtain average monthly losses.

4.  **Visualization (Logarithmic Scale):**
    * Creates a bar graph using Plotly Express to visualize the average monthly losses for each risk category.
    * The y-axis is set to a logarithmic scale to better represent data with significant variations in magnitude.

5.  **Output:**
    * Prints the calculated average monthly losses as a dictionary.

**Key Considerations:**

* The use of a logarithmic y-axis allows for better visualization of data where there are large differences in the magnitude of the values being compared.
* The 2.5% reduction applied to outlier related losses, is a hypothetical risk percentage, that should be adjusted to the proper value, if available.
* This model provides an estimation of potential losses based on defined risk factors. Actual losses may vary.

In [58]:
def estimate_average_monthly_losses(df, start_date, end_date, boleto_time_threshold_days=7):


    df['date_request_submitted'] = pd.to_datetime(df['date_request_submitted'], errors='coerce')
    df['date_request_received'] = pd.to_datetime(df['date_request_received'], errors='coerce')
    df = df.loc[(df['date_request_submitted'] >= start_date) & (df['date_request_submitted'] <= end_date)]

    def is_outlier(series):
        z_scores = stats.zscore(series, nan_policy='omit')
        return (abs(z_scores) > 1.96).any()

    losses = {
        'unknown_blank_status_losses': 0,
        'outlier_losses': 0,
        'self_recipient_losses': 0,
        'bel_deu_losses': 0,
    }

    unknown_blank_status_losses = df.loc[df['payment_reference_classification'].isin(['Other/unknown', 'blank']), 'invoice_value_usd'].sum() * 0.025 * 0.04
    losses['unknown_blank_status_losses'] = unknown_blank_status_losses

    df['is_outlier'] = df.groupby(pd.Grouper(key='date_request_submitted', freq='M'))['invoice_value_usd'].transform(is_outlier)
    outlier_losses = df.loc[df['is_outlier'], 'invoice_value_usd'].sum() * 0.025 * 0.04
    losses['outlier_losses'] = outlier_losses

    self_recipient_losses = df.loc[(df['transfer_to_self'] == 'Self-recipient: Name match') & (df['is_outlier']), 'invoice_value_usd'].sum() * 0.025 * 0.04
    losses['self_recipient_losses'] = self_recipient_losses

    bel_deu_losses = df.loc[((df['addr_country_code'] == 'BEL') | (df['addr_country_code'] == 'DEU')) & (df['is_outlier']), 'invoice_value_usd'].sum() * 0.025 * 0.04
    losses['bel_deu_losses'] = bel_deu_losses


    num_months = (pd.to_datetime(end_date) - pd.to_datetime(start_date)).days / 30.44

    average_monthly_losses = {k: v / num_months for k, v in losses.items()}

    plot_df = pd.DataFrame(list(average_monthly_losses.items()), columns=['Category', 'Average Monthly Loss'])

    fig = px.bar(
        plot_df,
        x='Category',
        y='Average Monthly Loss',
        title='Average Monthly Losses by Category (Logarithmic Y-Axis)',
        labels={'Average Monthly Loss': 'Average Monthly Loss (USD)'},
        log_y=True
    )

    fig.show()

    return average_monthly_losses

start_date = '2016-01-01'
end_date = '2017-12-31'
estimated_losses = estimate_average_monthly_losses(df_trimmed, start_date, end_date, boleto_time_threshold_days=14)
print(estimated_losses)



{'unknown_blank_status_losses': 1915.8434888279808, 'outlier_losses': 2422.3914684577076, 'self_recipient_losses': 108.38804639133058, 'bel_deu_losses': 158.04233520582196}


This is very rough estimation of losses based mostly on my undertanding of business and some assumptions. Apllying stricter checks for these types of transactions would probably cause 5K in USD in losses.

 **Implications:**

Prioritize strategies to address outlier transactions and transactions with unknown/blank payment reference classifications.
Investigate the underlying causes of outliers and unclear payment reference classifications to implement effective risk mitigation measures.
While "self_recipient_losses" and "bel_deu_losses" are lower, continue to monitor and evaluate these categories for potential increases in risk.

# What can we do?

## Enhanced Customer Information for Risk Assessment

Based on my analysis, here's the information should be gather to better assess customer risk and determine transaction trustworthiness:

**1. High-Risk Customer Due Diligence:**

* **New Users:**
    * **Purpose of Transaction:** Understand the reason, especially for large amounts or unusual destinations.
* **Outlier Transactions:**
    * **Unusual Behavior:** Analyze past transactions for sudden changes.
* **Unknown/Blank Payment Reference:**
    * **Clarify Purpose:** Request more information.
    * **Verify Identity:** Perform additional checks if needed.
* **Self-Recipient Transactions:**
    * **Account Activity:** Analyze for frequent self-transfers.
    * **Justification:** Request explanations, especially for large or frequent ones.
* **BEL/DEU Transactions:**
    * **Monitor Closely:** Implement stricter monitoring.
    * **Enhanced Scrutiny:** Consider extra verification for high-value or suspicious transactions.

**2. Data Collection Methods:**

* **Customer Questionnaires:** Design dynamic questionnaires based on risk profiles.
* **Third-Party Data Providers:** Partner with providers for enhanced due diligence.


### Trade-offs

## Trade-offs of Enhanced Customer Information and Risk Assessment Initiatives:

**1. Increased Friction for Users:**

* **More Information Required:** Requesting additional information from customers, especially new users or those flagged as high-risk, can increase friction and potentially deter some users.
* **Longer Processing Times:** Enhanced due diligence and verification checks can lead to longer processing times for transactions, impacting user experience.
* **Reduced User Anonymity:** Collecting more data about customers may raise privacy concerns and reduce the level of anonymity they expect.

**2. Implementation and Operational Costs:**

* **Technology Investment:** Implementing new tools and systems, such as automated KYC/AML checks or transaction monitoring systems, can require significant upfront investment.
* **Operational Overhead:** Manual review of suspicious transactions or customer information can increase operational costs and require additional staffing.
* **Maintenance and Updates:** Ongoing maintenance and updates of new systems and processes will incur ongoing costs.

**3. Potential for False Positives/Negatives:**

* **False Positives:** Stricter risk assessment measures may lead to legitimate transactions being flagged as suspicious, causing inconvenience for users and potentially damaging customer relationships.
* **False Negatives:** Despite enhanced measures, some fraudulent activities may still go undetected, leading to financial losses and reputational damage.

**4. Regulatory and Compliance Challenges:**

* **Data Privacy:** Collecting and storing sensitive customer data requires strict adherence to data privacy regulations, which can be complex and vary across jurisdictions.
* **Balancing Security and User Experience:** Finding the right balance between security measures and user experience can be challenging, as overly strict measures can deter legitimate users.

**5. Adapting to Evolving Risks:**

* **Dynamic Threat Landscape:** The methods used by fraudsters and criminals are constantly evolving, requiring continuous adaptation and improvement of risk assessment strategies.
* **Staying Ahead of the Curve:** Keeping up with new technologies and regulatory changes can be demanding and require ongoing investment in research and development.

**Mitigating Trade-offs:**

* **Transparency and Communication:** Clearly communicate to users why certain information is being requested and how it will be used to build trust and transparency.
* **Streamlined Processes:** Design user-friendly processes for data collection and verification to minimize friction and maintain a positive user experience.
* **Robust Technology:** Invest in reliable and efficient technology solutions to automate tasks and reduce manual intervention.
* **Continuous Monitoring and Improvement:** Regularly monitor the effectiveness of risk assessment measures and adapt strategies based on new data and insights.
* **Collaboration and Knowledge Sharing:** Foster collaboration with industry partners and regulatory bodies to stay informed about emerging risks and best practices.
